# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use free open source models on Ollama. I also use paid open-source models via Groq and OpenRouter. Only pick the models you want to!
            </span>
        </td>
    </tr>
</table>

In [56]:
# imports
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from my_system_info import retrieve_system_info
import re


In [57]:
load_dotenv(override=True)
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
hf_token = os.getenv('HF_TOKEN')

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

if hf_token:
    print(f"HF_TOKEN exists and begins {hf_token[:2]}")
else:
    print("HF_TOKEN not set (and this is optional)")



Google API Key exists and begins AI
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-or-
HF_TOKEN exists and begins hf


In [58]:
# Connect to client libraries

gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"
huggingface_url = "https://router.huggingface.co/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)
huggingface = OpenAI(api_key=hf_token, base_url=huggingface_url)


In [59]:
models = ["gemini-3.5-flash", "gemini-2.5-flash", "gemma-4-31b-it", "deepseek-ai/DeepSeek-V4-Pro",
          "deepseek-ai/DeepSeek-V4-Flash", "Qwen/Qwen2.5-Coder-3B-Instruct:nscale", "Qwen/Qwen3.5-397B-A17B:scaleway",
          "meta-llama/Llama-3.3-70B-Instruct", "openai/gpt-oss-20b", "openai/gpt-oss-120b", "qwen/qwen3-32b",
          "llama3.2:1b", "baidu/cobuddy:free"]

clients = {"gemini-3.5-flash": gemini, "gemini-2.5-flash": gemini, "gemma-4-31b-it": gemini,
           "deepseek-ai/DeepSeek-V4-Pro": huggingface, "deepseek-ai/DeepSeek-V4-Flash": huggingface,
           "Qwen/Qwen2.5-Coder-3B-Instruct:nscale": huggingface, "Qwen/Qwen3.5-397B-A17B:scaleway": huggingface,
           "meta-llama/Llama-3.3-70B-Instruct": huggingface, "openai/gpt-oss-20b": groq, "openai/gpt-oss-120b": groq,
           "qwen/qwen3-32b": groq, "llama3.2:1b": ollama,
           "baidu/cobuddy:free": openrouter, "qwen/qwen3-coder:free": openrouter}

# Want to keep costs ultra-low? Replace this with models of your choice, using the examples from yesterday

In [60]:
system_info = retrieve_system_info()
system_info

{'os': {'system': 'Darwin',
  'arch': 'arm64',
  'release': '24.5.0',
  'version': 'Darwin Kernel Version 24.5.0: Tue Apr 22 19:54:26 PDT 2025; root:xnu-11417.121.6~2/RELEASE_ARM64_T8112',
  'kernel': '24.5.0',
  'distro': None,
  'wsl': False,
  'rosetta2_translated': False,
  'target_triple': 'arm64-apple-darwin24.5.0'},
 'package_managers': ['xcode-select (CLT)'],
 'cpu': {'brand': 'Apple M2',
  'cores_logical': 8,
  'cores_physical': 8,
  'simd': []},
 'toolchain': {'compilers': {'gcc': 'Apple clang version 17.0.0 (clang-1700.0.13.5)',
   'g++': 'Apple clang version 17.0.0 (clang-1700.0.13.5)',
   'clang': 'Apple clang version 17.0.0 (clang-1700.0.13.5)',
   'msvc_cl': ''},
  'build_tools': {'cmake': '', 'ninja': '', 'make': 'GNU Make 3.81'},
  'linkers': {'ld_lld': ''}}}

## Overwrite this with the commands from yesterday

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [61]:
def build_compile_command(file):
    # `file` is the base name (without extension) located in the `output` directory
    output_dir = "output"
    src = os.path.join(output_dir, f"{file}.cpp")
    out = os.path.join(output_dir, file)
    compile_command = [
        "clang++",
        "-std=c++17",
        "-Ofast",
        "-mcpu=native",
        "-flto=thin",
        "-fvisibility=hidden",
        "-DNDEBUG",
        src,
        "-o",
        out,
    ]
    return compile_command

def build_run_command(file):
    # `file` is the base name (without extension) located in the `output` directory
    out = os.path.join("output", file)
    run_command = [out]
    return run_command


## And now, on with the main task

In [62]:
system_prompt = """
Your task is to convert Python code into high performance C++ code.
Respond only with C++ code. Do not provide any explanation other than occasional comments.
The C++ response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to C++ with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.cpp and then compiled and executed; the compilation command is:
{build_compile_command("main")}
Respond only with C++ code. No explanation other than occasional comments.
Python code to port:

```python
{python}
```
"""

In [63]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [64]:
def write_output(model, cpp):
    output_dir = "output"

    os.makedirs(output_dir, exist_ok=True)

    file_path = os.path.join(
        output_dir,
        f"{model}.cpp"
    )

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(cpp)

    return file_path

In [65]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```','')
    write_output(model, reply)
    return reply

In [66]:
pi = """
import time

def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations+1):
        j = i * param1 - param2
        result -= (1/j)
        j = i * param1 + param2
        result += (1/j)
    return result

start_time = time.time()
result = calculate(200_000_000, 4, 1) * 4
end_time = time.time()

print(f"Result: {result:.12f}")
print(f"Execution Time: {(end_time - start_time):.6f} seconds")
"""

In [67]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [68]:
def compile_and_run(model="main"):
    """Compile and run the generated C++ file from the `output` directory.

    model: base filename (without extension) stored under `output/<model>.cpp`.
    """
    try:
        # sanitize model name to match the filename written by write_output()
        safe_model = re.sub(r"[^A-Za-z0-9_.-]", "_", model)
        compile_cmd = build_compile_command(safe_model)
        subprocess.run(compile_cmd, check=True, text=True, capture_output=True)
        print("Compilation successful!\n")

        # Run the compiled binary 3 times
        for i in range(3):
            result = subprocess.run(build_run_command(safe_model), check=True, text=True, capture_output=True, timeout=30)
            print(f"Run {i+1} output:\n{result.stdout}")

    except subprocess.CalledProcessError as e:
        stderr = e.stderr if hasattr(e, 'stderr') and e.stderr else str(e)
        print(f"Compilation/Run error:\n{stderr}")
    except FileNotFoundError:
        print(f"Error: Compiler or binary not found. Ensure 'clang++' is installed and the source file exists in the 'output' directory.")
    except Exception as e:
        print(f"Unexpected error: {type(e).__name__}: {e}")

In [69]:
with gr.Blocks() as ui:
    with gr.Row():
        python = gr.Textbox(label="Python code:", lines=28, value=pi)
        cpp = gr.Textbox(label="C++ code:", lines=28)
    with gr.Row():
        model = gr.Dropdown(models, label="Select model", value=models[0])
        convert = gr.Button("Convert code")

    convert.click(port, inputs=[model, python], outputs=[cpp])

ui.launch(inbrowser=True,theme=gr.themes.Citrus())

* Running on local URL:  http://127.0.0.1:7905
* To create a public link, set `share=True` in `launch()`.


In [70]:
py_output = run_python(pi)
print(py_output)
compile_and_run("gemini-2.5-flash")

An error occurred:
clang++: error: no such file or directory: 'main.cpp.cpp'
clang++: error: no input files



Qwen 2.5 Coder: Fail  
DeepSeek Coder v2: 0.114050084  
OpenAI gpt-oss 20B: 0.080438  
Qwen 30B: 0.113734  
OpenAI gpt-oss 120B: 1.407383




In Ed's experiments, the performance speedups were:

9th place: Qwen 2.5 Coder: Fail  
8th place: OpenAI GPT-OSS 120B: 14X speedup    
7th place: DeepSeek Coder v2: 168X speedup  
6th place: Qwen3 Coder 30B: 168X speedup   
5th place: Claude Sonnet 4.5: 184X speedup   
4th place: GPT-5: 233X speedup  
**3rd place: oss-20B: 238X speedup**  
2nd place: Grok 4: 1060X speedup  
1st place: Gemini 2.5 Pro: 1440X speedup  